# CXR Disease Detection — Training Pipeline

Restart kernel and **Run All Cells** to train the three models in order:

1. **YOLOv8** — 10 epochs
2. **RT-DETR** — 20 epochs
3. **Faster R-CNN** — 20 epochs

Checkpoints are saved every 10 epochs.

In [7]:
%pip install -q ultralytics

In [8]:
import os
import glob
import random
import shutil
import yaml

from tqdm import tqdm
import pandas as pd
import numpy as np
import cv2

from google.colab import drive
from IPython.display import display

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader

import ultralytics
from ultralytics import YOLO, RTDETR

from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')

print(f"PyTorch     : {torch.__version__}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch     : 2.10.0+cu128
Ultralytics : 8.4.53
CUDA        : True
GPU         : Tesla T4


In [9]:
# Paths for Google Colab - data on Drive, intermediate dataset on local VM disk for speed
PROJECT_ROOT = "/content/drive/MyDrive/ObjectDetection_Project"
DATA_ROOT = os.path.join(PROJECT_ROOT, "vindr_data")
CSV_PATH = os.path.join(DATA_ROOT, "train.csv")
META_PATH = os.path.join(DATA_ROOT, "train_meta.csv")
IMG_SOURCE_DIR = os.path.join(DATA_ROOT, "train")

# Local VM disk - much faster I/O than Drive during training
YOLO_DATASET_DIR = "/content/yolo_dataset"
YAML_PATH = os.path.join(YOLO_DATASET_DIR, "data.yaml")

# Checkpoints saved to Drive for persistence
YOLO_RUN_DIR = os.path.join(PROJECT_ROOT, "YOLO_Runs")
RTDETR_RUN_DIR = os.path.join(PROJECT_ROOT, "Transformer_Runs")
FASTER_RCNN_SAVE_DIR = os.path.join(PROJECT_ROOT, "FasterRCNN_Runs")

YOLO_RUN_NAME = "yolov8n_cxr_test"
RTDETR_RUN_NAME = "rtdetr_cxr_test"

TARGET_CLASSES = [
    "Aortic enlargement",
    "Cardiomegaly",
    "Pleural effusion",
    "Pulmonary fibrosis",
    "Nodule/Mass",
]

for d in [YOLO_DATASET_DIR, YOLO_RUN_DIR, RTDETR_RUN_DIR, FASTER_RCNN_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Data source : {DATA_ROOT}")
print(f"YOLO dataset: {YOLO_DATASET_DIR}")
print(f"Classes     : {len(TARGET_CLASSES)}")


Data source : /content/drive/MyDrive/ObjectDetection_Project/vindr_data
YOLO dataset: /content/yolo_dataset
Classes     : 5


## Data Preparation

Converts VinDr-CXR annotations to YOLO format. Skipped if dataset already exists.

In [10]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"train.csv not found at: {CSV_PATH}")
if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"train_meta.csv not found at: {META_PATH}")

# Skip rebuild if dataset already exists and is non-trivial
train_imgs_dir = os.path.join(YOLO_DATASET_DIR, "images", "train")
if os.path.exists(train_imgs_dir) and len(os.listdir(train_imgs_dir)) > 100:
    print(f"YOLO dataset already exists at {YOLO_DATASET_DIR}, skipping rebuild.")
    print("Delete that folder to force a rebuild.")
else:
    df = pd.read_csv(CSV_PATH)
    meta_df = pd.read_csv(META_PATH).rename(columns={"dim0": "height", "dim1": "width"})
    df = pd.merge(df, meta_df[["image_id", "height", "width"]], on="image_id", how="left")

    # Drop annotations without metadata (would produce NaN in label files)
    missing_meta = df["width"].isna().sum()
    if missing_meta > 0:
        print(f"Warning: dropping {missing_meta} annotations missing metadata.")
    df = df.dropna(subset=["width", "height"]).copy()

    class_to_id = {cls_name: idx for idx, cls_name in enumerate(TARGET_CLASSES)}
    df_filtered = df[df["class_name"].isin(TARGET_CLASSES)].copy()
    df_filtered["class_id"] = df_filtered["class_name"].map(class_to_id).astype(int)

    # Filter degenerate bboxes (would crash YOLO label parsing)
    invalid_mask = (
        (df_filtered["x_max"] <= df_filtered["x_min"])
        | (df_filtered["y_max"] <= df_filtered["y_min"])
    )
    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        print(f"Warning: dropping {n_invalid} degenerate bboxes.")
    df_filtered = df_filtered[~invalid_mask].copy()

    # Pascal VOC -> YOLO format (normalized cx, cy, w, h)
    df_filtered["x_center"] = ((df_filtered["x_min"] + df_filtered["x_max"]) / 2) / df_filtered["width"]
    df_filtered["y_center"] = ((df_filtered["y_min"] + df_filtered["y_max"]) / 2) / df_filtered["height"]
    df_filtered["w"] = (df_filtered["x_max"] - df_filtered["x_min"]) / df_filtered["width"]
    df_filtered["h"] = (df_filtered["y_max"] - df_filtered["y_min"]) / df_filtered["height"]

    # Split by image to avoid leakage
    unique_images = df_filtered["image_id"].unique()
    train_imgs, val_imgs = train_test_split(unique_images, test_size=0.2, random_state=42)
    print(f"Total images: {len(unique_images)} | Train: {len(train_imgs)} | Val: {len(val_imgs)}")


    def build_dataset(image_ids, subset_name):
        img_dir = os.path.join(YOLO_DATASET_DIR, "images", subset_name)
        label_dir = os.path.join(YOLO_DATASET_DIR, "labels", subset_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)

        subset_df = df_filtered[df_filtered["image_id"].isin(image_ids)]
        grouped = subset_df.groupby("image_id")
        missing = 0

        for img_id, group in tqdm(grouped, desc=f"Building {subset_name}"):
            src_img = os.path.join(IMG_SOURCE_DIR, f"{img_id}.png")
            if not os.path.exists(src_img):
                missing += 1
                continue
            dst_img = os.path.join(img_dir, f"{img_id}.png")
            if not os.path.exists(dst_img):
                shutil.copy(src_img, dst_img)
            label_file = os.path.join(label_dir, f"{img_id}.txt")
            labels = group[["class_id", "x_center", "y_center", "w", "h"]].values
            np.savetxt(label_file, labels, fmt="%d %.6f %.6f %.6f %.6f")

        if missing > 0:
            print(f"  Warning: {missing} source images not found.")

    build_dataset(train_imgs, "train")
    build_dataset(val_imgs, "val")
    print(f"Dataset ready at: {YOLO_DATASET_DIR}")


Total images: 4285 | Train: 3428 | Val: 857


Building train: 100%|██████████| 3428/3428 [12:17<00:00,  4.65it/s]  


Building val: 100%|██████████| 857/857 [02:46<00:00,  5.13it/s] 

Dataset ready at: /content/yolo_dataset


In [11]:
# Use absolute paths so Ultralytics resolves them correctly regardless of cwd
data_yaml = {
    "train": os.path.abspath(os.path.join(YOLO_DATASET_DIR, "images", "train")),
    "val": os.path.abspath(os.path.join(YOLO_DATASET_DIR, "images", "val")),
    "nc": len(TARGET_CLASSES),
    "names": list(TARGET_CLASSES),
}

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml written: {YAML_PATH}")


data.yaml written: /content/yolo_dataset/data.yaml


## 1. YOLOv8 (10 epochs)

In [ ]:
# Saves best.pt, last.pt, and epoch10.pt thanks to save_period=10
yolo_model = YOLO("yolov8n.pt")

yolo_results = yolo_model.train(
    data=YAML_PATH,
    epochs=10,
    imgsz=512,
    batch=16,
    project=YOLO_RUN_DIR,
    name=YOLO_RUN_NAME,
    exist_ok=True,         # overwrite the same run folder so app.py finds it deterministically
    save_period=10,        # checkpoint every 10 epochs
    patience=0,            # disable early stopping for short runs
)

print("YOLOv8 training done.")
print(f"Best weights: {os.path.join(YOLO_RUN_DIR, YOLO_RUN_NAME, 'weights', 'best.pt')}")


Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_cxr_test, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

## 2. RT-DETR (20 epochs)

In [ ]:
# RT-DETR is heavier than YOLOv8 - smaller batch size helps fit in memory
rtdetr_model = RTDETR("rtdetr-l.pt")

rtdetr_results = rtdetr_model.train(
    data=YAML_PATH,
    epochs=20,
    imgsz=512,
    batch=8,
    project=RTDETR_RUN_DIR,
    name=RTDETR_RUN_NAME,
    exist_ok=True,
    save_period=10,
    patience=0,
)

print("RT-DETR training done.")
print(f"Best weights: {os.path.join(RTDETR_RUN_DIR, RTDETR_RUN_NAME, 'weights', 'best.pt')}")


Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_cxr_test, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/20      13.4G      1.145     0.7744     0.4239          7       1024: 100% ━━━━━━━━━━━━ 317/317 1.3s/it 6:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 3.4it/s 11.3s
                   all        608       3111      0.283      0.376      0.253      0.125

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/20      13.6G     0.6641     0.6051      0.151          7       1024: 100% ━━━━━━━━━━━━ 317/317 1.2s/it 6:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.2it/s 9.0s
                   all        608       3111      0.385       0.39      0.315      0.152

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      18/20      4.95G     0.4866     0.5358      0.195          7        512: 100% ━━━━━━━━━━━━ 317/317 1.9it/s 2:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.4it/s 8.7s
                   all        608       3111       0.45      0.457      0.368      0.216

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      19/20      4.95G      1.063     0.4409      0.331          8        512: 0% ──────────── 0/317  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      19/20      4.95G     0.4771      0.527     0.1898          7        512: 100% ━━━━━━━━━━━━ 317/317 1.9it/s 2:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.1it/s 9.2s
                   all        608       3111      0.433      0.461      0.372      0.218

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      20/20      4.95G     0.4407     0.4921     0.1656          8        512: 0% ──────────── 0/317  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      20/20      4.95G     0.4692     0.5137     0.1863          7        512: 100% ━━━━━━━━━━━━ 317/317 1.9it/s 2:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 4.6it/s 8.2s
                   all        608       3111      0.435      0.466      0.355      0.206

20 epochs completed in 1.620 hours.
Optimizer stripped from /content/drive/MyDrive/ObjectDetection_Project/Transformer_Runs/rtdetr_cxr_test/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/ObjectDetection_Project/Transformer_Runs/rtdetr_cxr_test/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/ObjectDetection_Project/Transformer_Runs/rtdetr_cxr_test/weights/best.pt...
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,994,015 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━

## 3. Faster R-CNN (20 epochs)

In [12]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {device}")


class CXRDataset(Dataset):
    """Faster R-CNN dataset that reads YOLO labels and converts to Pascal VOC format."""

    def __init__(self, img_dir, label_dir, img_size=512):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.img_size = img_size
        self.imgs = sorted(
            [f for f in os.listdir(img_dir) if f.lower().endswith(".png")]
        )

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Resize image to match the scale of decoded bboxes
        img_resized = cv2.resize(
            img_rgb,
            (self.img_size, self.img_size),
            interpolation=cv2.INTER_LINEAR,
        )

        # from_numpy is zero-copy; faster than torch.tensor()
        img = img_resized.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1).contiguous()

        label_path = os.path.join(
            self.label_dir,
            os.path.splitext(img_name)[0] + ".txt",
        )
        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    class_id, cx, cy, w, h = map(float, parts)

                    x_min = (cx - w / 2) * self.img_size
                    y_min = (cy - h / 2) * self.img_size
                    x_max = (cx + w / 2) * self.img_size
                    y_max = (cy + h / 2) * self.img_size

                    # Clip to image bounds
                    x_min = max(0.0, min(x_min, self.img_size - 1.0))
                    y_min = max(0.0, min(y_min, self.img_size - 1.0))
                    x_max = max(0.0, min(x_max, self.img_size - 1.0))
                    y_max = max(0.0, min(y_max, self.img_size - 1.0))

                    # Drop degenerate boxes (cause NaN in RPN loss)
                    if x_max <= x_min or y_max <= y_min:
                        continue

                    boxes.append([x_min, y_min, x_max, y_max])
                    # +1 because class_id 0 is reserved for background
                    labels.append(int(class_id) + 1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


def create_faster_rcnn_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


train_dataset = CXRDataset(
    os.path.join(YOLO_DATASET_DIR, "images", "train"),
    os.path.join(YOLO_DATASET_DIR, "labels", "train"),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=2,
)

val_dataset = CXRDataset(
    os.path.join(YOLO_DATASET_DIR, "images", "val"),
    os.path.join(YOLO_DATASET_DIR, "labels", "val"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=2,
)

# 5 disease classes + 1 background
faster_rcnn = create_faster_rcnn_model(num_classes=len(TARGET_CLASSES) + 1).to(device)

print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")
print("Faster R-CNN ready.")


Device: cuda
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 186MB/s]


Train images: 2535 | Val images: 608
Faster R-CNN ready.


In [13]:
params = [p for p in faster_rcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1) # Reduce LR by 10x every 3 epochs

NUM_EPOCHS = 20
CHECKPOINT_EVERY = 10
GRAD_CLIP_NORM = 10.0
PATIENCE = 5 # Number of epochs to wait before stopping if validation loss doesn't improve
best_val_loss = float("inf")
patience_counter = 0

best_path = os.path.join(FASTER_RCNN_SAVE_DIR, "best_faster_rcnn.pth")
last_path = os.path.join(FASTER_RCNN_SAVE_DIR, "last_faster_rcnn.pth")

print(f"Training Faster R-CNN for {NUM_EPOCHS} epochs...\n" + "=" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING PHASE ---
    faster_rcnn.train()
    train_loss_epoch = 0.0
    num_train_batches = 0

    train_loop = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Training")
    for i, (images, targets) in enumerate(train_loop):
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [
            {k: v.to(device, non_blocking=True) for k, v in t.items()}
            for t in targets
        ]

        loss_dict = faster_rcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Skip NaN/Inf batches instead of poisoning the epoch
        if not torch.isfinite(losses):
            optimizer.zero_grad(set_to_none=True)
            del images, targets, loss_dict, losses
            continue

        optimizer.zero_grad(set_to_none=True)
        losses.backward()
        torch.nn.utils.clip_grad_norm_(faster_rcnn.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()

        train_loss_epoch += losses.item()
        num_train_batches += 1
        train_loop.set_postfix(loss=losses.item())
        del images, targets, loss_dict, losses

    avg_train_loss = train_loss_epoch / max(num_train_batches, 1)

    # --- VALIDATION PHASE ---
    # torchvision FasterRCNN only returns loss_dict in .train() mode.
    # .eval() mode returns predictions instead. Keep .train() + no_grad.
    faster_rcnn.train()
    val_loss_epoch = 0.0
    num_val_batches = 0

    val_loop = tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Validation")
    with torch.no_grad():
        for images, targets in val_loop:
            images = [img.to(device, non_blocking=True) for img in images]
            targets = [
                {k: v.to(device, non_blocking=True) for k, v in t.items()}
                for t in targets
            ]
            loss_dict = faster_rcnn(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if torch.isfinite(losses):
                val_loss_epoch += losses.item()
                num_val_batches += 1
                val_loop.set_postfix(loss=losses.item())
            del images, targets, loss_dict, losses

    avg_val_loss = val_loss_epoch / max(num_val_batches, 1)

    print(
        f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
        f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}"
    )

    # Update LR scheduler
    lr_scheduler.step()

    # --- EARLY STOPPING AND CHECKPOINTING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(faster_rcnn.state_dict(), best_path)
        print(f"  -> New best (val={best_val_loss:.4f}) saved.")
        patience_counter = 0 # Reset patience since loss improved
    else:
        patience_counter += 1
        print(f"  -> Validation loss did not improve. Patience: {patience_counter}/{PATIENCE}")

    # Periodic checkpoint every CHECKPOINT_EVERY epochs
    if epoch % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(FASTER_RCNN_SAVE_DIR, f"epoch_{epoch}_faster_rcnn.pth")
        torch.save(faster_rcnn.state_dict(), ckpt_path)
        print(f"  -> Checkpoint at epoch {epoch}.")

    if patience_counter >= PATIENCE:
        print(f"  -> Early stopping triggered after {patience_counter} epochs without improvement.")
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save final state regardless of validation performance (though early stopping might have already saved the best)
torch.save(faster_rcnn.state_dict(), last_path)
print("=" * 50)
print(f"Faster R-CNN training done.")
print(f"  Best: {best_path}")
print(f"  Last: {last_path}")

Training Faster R-CNN for 20 epochs...


Epoch 1/20 Validation: 100%|██████████| 152/152 [00:55<00:00,  2.75it/s, loss=0.448]


Epoch   1/20 | Train: 0.5225 | Val: 0.4519
  -> New best (val=0.4519) saved.


Epoch 2/20 Validation: 100%|██████████| 152/152 [00:55<00:00,  2.75it/s, loss=0.426]


Epoch   2/20 | Train: 0.4498 | Val: 0.4495
  -> New best (val=0.4495) saved.


...

Epoch   7/20 | Train: 0.3519 | Val: 0.4367
  -> Validation loss did not improve. Patience: 3/5


Epoch 8/20 Validation: 100%|██████████| 152/152 [00:55<00:00,  2.75it/s, loss=0.446]


Epoch   8/20 | Train: 0.3503 | Val: 0.4386
  -> Validation loss did not improve. Patience: 4/5


Epoch 9/20 Validation: 100%|██████████| 152/152 [00:55<00:00,  2.75it/s, loss=0.443]


Epoch   9/20 | Train: 0.3493 | Val: 0.4385
  -> Validation loss did not improve. Patience: 5/5
  -> Early stopping triggered after 5 epochs without improvement.
Faster R-CNN training done.
  Best: /content/drive/MyDrive/ObjectDetection_Project/FasterRCNN_Runs/best_faster_rcnn.pth
  Last: /content/drive/MyDrive/ObjectDetection_Project/FasterRCNN_Runs/last_faster_rcnn.pth
